In [ ]:

from pathlib import Path
from collections import Counter
from PIL import Image

# Find project root
current_dir = Path.cwd().resolve()

ROOT_DIR = next(
    (p for p in [current_dir, *current_dir.parents]
     if (p / "config.py").exists()),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root not found!")

# Dataset paths
DATASET_DIR = ROOT_DIR / "data" / "raw" / "Kvasir-SEG"

IMAGE_DIR = DATASET_DIR / "images"
MASK_DIR = DATASET_DIR / "masks"

# Collect image and mask paths
extensions = {".jpg", ".jpeg", ".png"}

images = {
    p.stem: p
    for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in extensions
}

masks = {
    p.stem: p
    for p in MASK_DIR.iterdir()
    if p.suffix.lower() in extensions
}

# Check dataset counts
print("Total Images:", len(images))
print("Total Masks:", len(masks))

# Check matching files
missing_masks = set(images) - set(masks)
missing_images = set(masks) - set(images)

print("Missing Masks:", len(missing_masks))
print("Missing Images:", len(missing_images))

# Check dimensions
image_sizes = Counter()
size_mismatches = []

for name in sorted(set(images) & set(masks)):

    with Image.open(images[name]) as img:
        image_size = img.size

    with Image.open(masks[name]) as mask:
        mask_size = mask.size

    image_sizes[image_size] += 1

    if image_size != mask_size:
        size_mismatches.append(name)

print("Size Mismatches:", len(size_mismatches))

print("\nImage Size Distribution:")

for size, count in image_sizes.most_common():
    print(f"{size}: {count} images")

# Final validation
assert len(missing_masks) == 0
assert len(missing_images) == 0
assert len(size_mismatches) == 0

print("\nDataset validation successful!")

In [ ]:

import sys
import numpy as np

# Add project root to Python path
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.dataset import preprocess_image_mask

# Select a sample
sample_name = sorted(images.keys())[0]

image, mask = preprocess_image_mask(
    images[sample_name],
    masks[sample_name]
)

# Check results
print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("Image dtype:", image.dtype)
print("Mask dtype:", mask.dtype)

print("Image range:", image.min(), image.max())
print("Mask unique values:", np.unique(mask))

assert image.shape == (128, 128, 3)
assert mask.shape == (128, 128)
assert 0 <= image.min() <= image.max() <= 1
assert set(np.unique(mask)).issubset({0.0, 1.0})

print("\nPreprocessing successful!")

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split

from config import SEED, SPLITS_FILE

# Get matching image and mask IDs
image_ids = sorted(set(images) & set(masks))

# Ensure dataset is complete
assert len(images) == len(masks) == len(image_ids) == 1000

# Split: 80% Train, 20% Remaining
train_ids, remaining_ids = train_test_split(
    image_ids,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

# Split remaining into Validation and Test
val_ids, test_ids = train_test_split(
    remaining_ids,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

# Ensure no overlap between splits
assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)

# Build split records
records = []

for split_name, ids in [
    ("train", train_ids),
    ("val", val_ids),
    ("test", test_ids)
]:
    for image_id in ids:
        records.append({
            "image_id": image_id,
            "image_path": images[image_id].relative_to(
                ROOT_DIR
            ).as_posix(),
            "mask_path": masks[image_id].relative_to(
                ROOT_DIR
            ).as_posix(),
            "split": split_name
        })

# Create DataFrame
new_df = pd.DataFrame(records)

# Save once; preserve existing splits on reruns
SPLITS_FILE.parent.mkdir(parents=True, exist_ok=True)

if not SPLITS_FILE.exists():
    new_df.to_csv(SPLITS_FILE, index=False)
    print("New splits.csv created!")
else:
    print("Existing splits.csv preserved!")

# Load saved splits
df = pd.read_csv(SPLITS_FILE)

# Validate saved splits
assert df["image_id"].is_unique
assert set(df["image_id"]) == set(image_ids)
assert df["split"].value_counts().to_dict() == {
    "train": 800,
    "val": 100,
    "test": 100
}

print("\nDataset Split Distribution:")
print(df["split"].value_counts())

print("\nTotal samples:", len(df))
print("Split file:", SPLITS_FILE)

print("\nDataset splitting successful!")

In [ ]:

import hashlib
import pandas as pd
from collections import defaultdict

from config import ROOT_DIR, SPLITS_FILE

# Load saved dataset splits
df = pd.read_csv(SPLITS_FILE)

# Check that image IDs are unique
assert df["image_id"].is_unique

# Store file hashes and their splits
hash_records = defaultdict(list)
missing_files = []

for row in df.itertuples(index=False):

    image_path = ROOT_DIR / row.image_path
    mask_path = ROOT_DIR / row.mask_path

    # Check file existence
    if not image_path.is_file():
        missing_files.append(str(image_path))
        continue

    if not mask_path.is_file():
        missing_files.append(str(mask_path))

    # Calculate SHA-256 hash
    file_hash = hashlib.sha256(
        image_path.read_bytes()
    ).hexdigest()

    hash_records[file_hash].append(
        (row.image_id, row.split)
    )

# Find exact duplicates
duplicate_groups = [
    records
    for records in hash_records.values()
    if len(records) > 1
]

# Find duplicates across different splits
cross_split_duplicates = [
    records
    for records in duplicate_groups
    if len({split for _, split in records}) > 1
]

print("Missing files:", len(missing_files))
print("Exact duplicate groups:", len(duplicate_groups))
print("Cross-split duplicates:", len(cross_split_duplicates))

if cross_split_duplicates:
    print("\nExamples:")
    for group in cross_split_duplicates[:5]:
        print(group)
else:
    print("\nNo exact cross-split duplicates found!")

In [ ]:

import json

from config import DATASET_DIR

metadata_path = DATASET_DIR / "kavsir_bboxes.json"

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Metadata type:", type(metadata).__name__)

if isinstance(metadata, dict):
    print("Number of entries:", len(metadata))

    # Show one example
    first_key = next(iter(metadata))

    print("\nSample key:", first_key)
    print("Sample value:")
    print(metadata[first_key])

elif isinstance(metadata, list):
    print("Number of entries:", len(metadata))
    print("\nSample entry:", metadata[0])

In [ ]:

import torch
import importlib
import src.dataset as dataset_module

# Reload updated dataset.py
importlib.reload(dataset_module)

PolypDataset = dataset_module.PolypDataset

# Select training samples
train_df = df[df["split"] == "train"]

# Create dataset
train_dataset = PolypDataset(train_df)

# Get first sample
image, mask = train_dataset[0]

print("Dataset size:", len(train_dataset))

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

print("Image dtype:", image.dtype)
print("Mask dtype:", mask.dtype)

print("Mask values:", torch.unique(mask))

# Validate outputs
assert len(train_dataset) == 800
assert image.shape == (3, 128, 128)
assert mask.shape == (1, 128, 128)
assert image.dtype == torch.float32
assert mask.dtype == torch.float32
assert torch.all((image >= 0) & (image <= 1))
assert torch.all((mask == 0) | (mask == 1))

print("\nDataset test successful!")

In [ ]:

from torch.utils.data import DataLoader

from config import BATCH_SIZE, SEED

# Split dataframes
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]

# Create datasets
train_dataset = PolypDataset(train_df)
val_dataset = PolypDataset(val_df)
test_dataset = PolypDataset(test_df)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:

# Get one training batch
images, masks = next(iter(train_loader))

print("Images shape:", images.shape)
print("Masks shape:", masks.shape)

print("Images dtype:", images.dtype)
print("Masks dtype:", masks.dtype)

print("Image min:", images.min().item())
print("Image max:", images.max().item())

print("Mask values:", torch.unique(masks))

# Validate batch
assert images.shape == (BATCH_SIZE, 3, 128, 128)
assert masks.shape == (BATCH_SIZE, 1, 128, 128)

assert images.dtype == torch.float32
assert masks.dtype == torch.float32

assert torch.all((images >= 0) & (images <= 1))
assert torch.all((masks == 0) | (masks == 1))

print("\nDataLoader test successful!")

In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

ROOT_DIR = next(
    (
        p for p in [current_dir, *current_dir.parents]
        if (p / "config.py").exists()
    ),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("Project root not found!")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("Project root:", ROOT_DIR)
print("Source directory exists:", (ROOT_DIR / "src").exists())

import src.dataset as dataset_module

print("Dataset module imported successfully!")

In [ ]:

import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

from config import SPLITS_FILE, IMAGE_SIZE

import src.dataset as dataset_module
importlib.reload(dataset_module)

# Load dataset splits
df = pd.read_csv(SPLITS_FILE)
train_df = df[df["split"] == "train"]

# Create datasets
original_dataset = dataset_module.PolypDataset(train_df)
augmented_dataset = dataset_module.AugmentedPolypDataset(train_df)

# Select one sample
sample_idx = 0

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):

    if i == 0:
        image, mask = original_dataset[sample_idx]
        title = "Original"
    else:
        image, mask = augmented_dataset[sample_idx]
        title = f"Augmented {i}"

    # Validate tensors
    assert image.shape == (3, IMAGE_SIZE, IMAGE_SIZE)
    assert mask.shape == (1, IMAGE_SIZE, IMAGE_SIZE)
    assert torch.all((image >= 0) & (image <= 1))
    assert torch.all((mask == 0) | (mask == 1))

    image_np = image.permute(1, 2, 0).numpy()
    mask_np = mask.squeeze(0).numpy()

    # Display image
    axes[0, i].imshow(image_np)
    axes[0, i].set_title(title)

    # Display image with mask overlay
    axes[1, i].imshow(image_np)
    axes[1, i].imshow(
        np.ma.masked_where(mask_np == 0, mask_np),
        cmap="Reds",
        alpha=0.5,
        vmin=0,
        vmax=1
    )
    axes[1, i].set_title(f"{title} + Mask")

    for row in range(2):
        axes[row, i].axis("off")

plt.tight_layout()
plt.show()

print("Augmentation test successful!")